# 1-2 LLM을 사용하며 알아야 할 것들 (버전 B)

| 항목 | 내용 |
|---|---|
| 원본 PDF | `docs/raw/Chapter01/day01/1-2-development-environment-and-llm-api-mechanism.pdf` (19쪽) |
| 원본 코드 | `src/raw/Chapter01/01-llm-api/`의 `01_requests.py`, `02_openai.py`, `03_langchain.py` |
| 실행 조건 | 실제 API를 호출합니다. 사외는 `LLM_PROVIDER=openai`, 사내는 `LLM_PROVIDER=samsung`입니다. |
| 버전 뜻 | A = 슬라이드 문구만, B = 슬라이드 요약 + 코드 연결, C = B + 왜 배우는지 |

In [ ]:
import json

from jupyterquiz import display_quiz
from assets.widgets import run_step, show_checklist, show_code_map, show_order_quiz, show_slides

In [ ]:
show_slides("1-2", [1, 2])

이 노트북은 "LLM을 사용하며 알아야 할 것들"을 네 장으로 나눠 봅니다(p1, p2).
네 장은 LLM의 학습과 추론, Tokenization(토큰 나누기), Context Window(한 번에 읽는 범위), LLM API의 대화 구조입니다(p2).
장 사이사이에 원본 실습 파일 세 개(`01_requests.py`, `02_openai.py`, `03_langchain.py`)를 한 단계씩 실행합니다.
세 파일은 모두 "LLM이 무엇인지 3문장으로 설명해줘"라는 같은 질문을 서로 다른 방법으로 보냅니다.
아래 코드 셀은 어느 LLM으로 연결되는지(`LLM_PROVIDER`)만 확인합니다. 기본값은 사내 게이트웨이(`samsung`)입니다.

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))  # 프로젝트의 .env 파일을 찾아 읽습니다

provider = os.getenv("LLM_PROVIDER", "samsung").strip().lower()
print("LLM_PROVIDER:", provider)

# 값은 출력하지 않습니다. 설정되어 있는지(True/False)만 봅니다.
필요한_이름 = {
    "openai": ["OPENAI_API_KEY", "OPENAI_MODEL"],
    "samsung": ["api_base_url", "model", "credential_key", "send_system_name", "user_id"],
}
for name in 필요한_이름.get(provider, []):
    print(f"{name} 설정됨:", bool(os.getenv(name)))

import config.llm
print("config 불러오기 성공")

**볼 지점**
- `LLM_PROVIDER`가 `openai` 또는 `samsung`으로 나오는지 확인합니다.
- 이름 옆이 모두 `True`인지 확인합니다. `False`가 있으면 `.env`에 그 값이 비어 있다는 뜻이니 강사에게 알립니다.
- 마지막 줄에 "config 불러오기 성공"이 나오면 준비가 끝났습니다.
- 키 값은 화면에 나오지 않습니다. 혹시 나오면 강사에게 알립니다.

## 1. LLM의 학습과 추론
### 1-1 사전 학습과 Instruction Tuning

슬라이드를 보기 전에 먼저 예측해 봅니다. 답은 바로 뒤의 p4에 있습니다.

In [ ]:
display_quiz("assets/quizzes/1-2/q11.json", max_width=800)

In [ ]:
show_slides("1-2", [3, 4])

LLM은 먼저 아주 많은 글(웹 문서, 책, 코드, 대화)에서 말의 패턴을 배웁니다. 이것을 사전 학습(Pre-training)이라고 합니다(p4).
이 단계의 목표는 지금까지의 글 다음에 올 가능성이 높은 토큰(글의 작은 조각)을 맞히는 것입니다(p4).
그래서 사전 학습만 한 모델은 지시를 따르기보다 그럴듯하게 이어 씁니다.
"To be or not to be" 뒤에 "that is the question"이 붙는 예시가 그 모습입니다(p4).
"다음 토큰을 맞힌다"는 생각은 p6의 생성 과정에서 다시 나옵니다(p6).

In [ ]:
show_slides("1-2", [5])

Instruction Tuning(지시 따르기 조정)은 질문과 답, 명령과 수행 결과 같은 예시로 모델을 한 번 더 학습시키는 과정입니다(p5).
이렇게 하면 모델이 입력을 "이어 쓸 문장"이 아니라 "해야 할 요청"으로 읽게 됩니다.
p5의 예시 데이터는 `instruction`(지시), `input`(입력), `output`(바라는 답) 세 칸으로 되어 있습니다(p5).
우리가 API나 챗봇으로 쓰는 모델은 대체로 이 과정을 거친 모델입니다(p5).
그래서 01_requests.py의 33줄처럼 "3문장으로 설명해줘"라고 지시만 적어도 답이 옵니다.

사전 학습은 다음 토큰을 맞히며 이어 쓰기를 배우고, Instruction Tuning은 그 위에 요청에 답하는 행동을 더합니다(p4, p5).
실습에서 호출하는 API 모델은 대체로 두 과정을 모두 거친 모델입니다(p5).

### 1-2 다음 토큰 예측과 생성 과정

In [ ]:
show_slides("1-2", [6])

LLM은 답을 한 번에 꺼내지 않고, 토큰을 하나씩 골라 붙이며 문장을 만듭니다(p6).
지금까지의 글을 보고 다음에 올 토큰 후보의 확률(나올 가능성)을 계산합니다.
그중 하나를 골라 글 뒤에 붙이고 이것을 되풀이하면 단어, 문장, 답변이 완성됩니다(p6).
이 "조금씩 붙는 모습"은 뒤의 Streaming(답을 조각으로 받는 방식) 실행에서 눈으로 봅니다(실행 ②와 실행 ③의 Streaming 부분).
후보를 어떻게 고를지 조절하는 설정은 p19의 temperature와 top-p입니다(p19).

**먼저 예측해 봅니다.** 다음 쪽을 보기 전에, 요청을 보내고 응답을 받기까지의 순서를 맞혀 보세요.

In [ ]:
show_order_quiz(**json.load(open("assets/quizzes/1-2/order12.json", encoding="utf-8")))

In [ ]:
show_slides("1-2", [7])

API(프로그램끼리 주고받는 창구)의 응답은 p6의 반복 생성을 서비스로 감싼 결과입니다(p7).
순서는 네 단계입니다. 요청 보내기 → 다음 토큰 후보 계산 → 토큰이 쌓여 답이 됨 → 종료 조건에서 응답 반환(p7).
1단계의 "메시지와 설정값"은 01_requests.py 32~35줄의 `messages`와 `temperature`입니다.
1단계의 "보낸다"는 45줄의 `requests.post(...)`이고, 4단계의 "반환"은 49줄의 `new_response.json()`으로 받는 부분입니다.
2·3단계는 서버 안에서 일어나므로 코드에는 보이지 않습니다(p7).

LLM은 토큰을 하나씩 예측해 붙이고, API는 그 과정을 "요청 → 계산 → 누적 → 반환" 네 단계로 감쌉니다(p6, p7).

### ▶ 실행 ① `01_requests.py` — 요청을 보내고 응답 받기

이 파일은 p7의 네 단계 중 1단계(요청 보내기)와 4단계(응답 받기)를 코드로 보여 줍니다(p7).
전용 도구 없이 `requests`(웹 요청을 보내는 파이썬 라이브러리, 즉 남이 만들어 둔 코드 묶음)로 직접 보내므로, 요청과 응답의 모양이 그대로 드러납니다.
요청 본문은 `payload`라는 딕셔너리(이름표가 붙은 값 묶음)입니다(30~37줄).
응답으로 받은 JSON(이름표가 붙은 값 묶음을 글자로 적은 형식) 전체와 그 안의 답변 텍스트를 차례로 출력합니다(51~54줄).
파일 끝 `"""` 사이의 코드(57~83줄)는 참고용이라 실행되지 않습니다.

In [ ]:
show_code_map('../src/raw/Chapter01/01-llm-api/01_requests.py', [{'title': '준비와 환경 분기',
  'marker': 'import os',
  'note': '필요한 도구를 불러오고 .env를 읽습니다.\n'
          'LLM_PROVIDER 값에 따라 요청 주소, 모델, 인증 헤더를 정합니다.\n'
          '키는 헤더에만 넣고 화면에 출력하지 않습니다.'},
 {'title': '요청 본문',
  'marker': 'payload = {',
  'page': 7,
  'note': 'p7의 1단계, 메시지와 설정값을 담는 곳입니다.\nmessages 안의 user 한 줄이 보낼 질문입니다.\ntemperature 0.5가 설정값입니다.'},
 {'title': '환경·모델별 조정',
  'marker': "if provider == 'samsung':",
  'page': 19,
  'note': '사내(samsung)이면 include_reasoning False를 추가합니다.\n'
          '그렇지 않고 모델이 gpt-6-luna이면 temperature를 빼고 reasoning_effort를 추가합니다.\n'
          '사내 경로에서는 temperature 0.5가 그대로 전송됩니다.'},
 {'title': '전송',
  'marker': 'new_response = requests.post',
  'page': 7,
  'note': '요청을 보냅니다(requests.post).\n'
          '서버가 오류로 답하면 raise_for_status()에서 멈춥니다.\n'
          '받은 응답을 JSON(데이터를 주고받는 텍스트 형식)으로 바꿉니다.'},
 {'title': '응답 읽기',
  'marker': 'print("### 응답 Json")',
  'page': 10,
  'note': '응답 JSON 전체를 먼저 출력합니다. usage가 있으면 여기서 토큰 수가 보입니다.\n'
          '그다음 choices[0].message.content에서 답변 텍스트만 꺼내 출력합니다.'},
 {'title': '참고 (실행 안 함)',
  'marker': '"""',
  'note': '""" 안의 코드는 실행되지 않는 참고용입니다.\nResponses API 주소로 보내는 예시입니다.'}], deck="1-2")

실행하기 전에 예측해 봅니다. 답은 단계 3의 출력에서 직접 확인합니다.

In [ ]:
display_quiz("assets/quizzes/1-2/q_run1.json", max_width=800)

#### 실행 ① · 단계 1

이 단계는 어디로(주소), 어떤 모델로, 어떤 인증으로 보낼지를 정합니다.
`openai`이면 OpenAI 주소와 키를, `samsung`이면 `config/samsung.py`의 설정을 쓰고, 둘 다 아니면 바로 오류를 내고 멈춥니다.
키 같은 인증 정보는 `headers`(헤더, 요청에 함께 붙여 보내는 부가 정보)에 들어갑니다.
여기서 만든 `api_base_url`, `model`, `headers`는 뒤의 요청 본문과 전송 단계에서 씁니다.

In [ ]:
ns01 = run_step('../src/raw/Chapter01/01-llm-api/01_requests.py', None, 'payload = {')

**볼 지점**
- 이 단계는 화면에 아무것도 출력하지 않습니다. 오류 없이 끝나면 준비가 된 것입니다.
- 9줄에서 `LLM_PROVIDER` 값을 읽고, 10~28줄에서 그 값에 따라 주소·모델·헤더를 정합니다.
- 오류가 나면 메시지를 읽습니다. "OPENAI_MODEL과 OPENAI_API_KEY가 필요합니다"(15줄)나 "사내 API 설정이 부족합니다"가 나오면 `.env`에서 메시지에 적힌 이름을 확인합니다.
- 키는 헤더에만 넣습니다(16~19줄, 26줄). 화면에는 나오지 않습니다.

#### 실행 ① · 단계 2

`payload`가 p7 1단계의 "메시지와 설정값"입니다(p7).
`messages`에는 역할이 `user`인 메시지 하나만 들어 있습니다. 역할 이야기는 p18에서 다시 합니다(p18).
`temperature`는 답이 얼마나 흔들릴지 정하는 값입니다. p19에서 다시 봅니다(p19).
사외에서 모델이 `gpt-6-luna`이면 이 값은 빠진 채 전송됩니다(40~43줄). 어느 모델이었는지는 다음 단계 출력의 `'model'`에서 확인합니다.

In [ ]:
ns01 = run_step('../src/raw/Chapter01/01-llm-api/01_requests.py', 'payload = {', 'new_response = requests.post')

**볼 지점**
- 이 단계도 출력이 없습니다. 위 코드 화면에서 확인합니다.
- `"messages"` 안의 `"role": "user"` 한 줄(33줄)이 우리가 보낼 메시지입니다.
- `"temperature": 0.5`(35줄)가 설정값입니다.
- 그 아래 `if`·`elif`(38~43줄)는 환경(`samsung`)과 모델(`gpt-6-luna`)에 따라 설정값을 조금 바꿉니다. 사외 실행에서 모델이 `gpt-6-luna`이면 42줄에서 temperature가 빠진 채 전송되고, 사내 경로에서는 0.5가 그대로 전송됩니다(38~43줄).

#### 실행 ① · 단계 3

`requests.post(...)`로 요청을 보내는 순간이 p7의 1단계입니다(p7).
`raise_for_status()`는 서버가 오류로 답하면 여기서 멈추게 합니다.
`new_response.json()`으로 받은 `recv_json`이 p7의 4단계 "최종 답변"입니다(p7).

In [ ]:
ns01 = run_step('../src/raw/Chapter01/01-llm-api/01_requests.py', 'new_response = requests.post', '"""')

**볼 지점**
- "### 결과 메시지" 아래 줄이 모델의 답입니다. 코드는 `choices[0].message.content`를 꺼내 출력했습니다(53~54줄).
- "### 응답 Json" 아래에서 `choices` → `message` → `content`를 따라가면 같은 텍스트가 있습니다.
- "### 응답 Json" 아래에서 `usage`를 찾습니다. 입력 토큰 수와 출력 토큰 수가 들어 있습니다 (p10).
- 이름의 뜻: `prompt_tokens` = 입력 토큰, `completion_tokens` = 출력 토큰, `total_tokens` = 둘의 합입니다.
- 숫자는 실행할 때마다, 환경마다 다릅니다. 출력에 적힌 값을 그대로 읽습니다.
- `usage`가 없으면 "이 환경에서는 제공되지 않음"으로 적습니다.

p7의 1단계는 `payload`를 만들어 `requests.post`로 보내는 줄이고, 4단계는 `new_response.json()`으로 받는 줄입니다(p7).
2·3단계(후보 계산, 누적)는 서버 안에서 일어나서 코드에 보이지 않습니다(p7).

## 2. Tokenization
### 2-1 토큰의 의미

In [ ]:
show_slides("1-2", [8, 9])

Tokenization(토큰화)은 사람의 말을 모델이 다룰 수 있는 작은 단위로 나누는 과정이고, 그 단위를 토큰이라고 합니다(p9).
토큰은 늘 단어 하나와 같지는 않고 단어 일부나 공백, 기호, 숫자 조각일 수도 있습니다.
한국어는 조사와 어미 때문에 더 잘게 나뉠 수 있습니다(p9).
p9 예시에서 "LangChain"은 "Lang"과 "Chain"으로, "만든다"는 "만든"과 "다"로 나뉩니다(p9).
입력한 질문과 모델의 답은 모두 토큰 단위로 셉니다. 이 숫자가 다음 쪽의 비용과 이어집니다(p10).

In [ ]:
display_quiz("assets/quizzes/1-2/q21.json", max_width=800)

토큰은 모델이 글을 읽고 쓰는 기본 단위이고, 단어와 같지 않으며 한국어는 더 잘게 나뉠 수 있습니다(p9).

### 2-2 토큰 수와 비용

In [ ]:
show_slides("1-2", [10])

LLM API의 비용과 처리량은 보통 입력 토큰 수와 출력 토큰 수로 정해집니다(p10).
입력 토큰에는 질문뿐 아니라 시스템 프롬프트(모델에게 주는 기본 지시)와 이전 대화도 모두 들어갑니다.
출력 토큰은 모델이 새로 만든 답의 길이이고, 생각하는 과정이 있는 모델은 그 생각에 쓴 토큰도 들어갑니다(p10).
그래서 프롬프트와 답이 길수록 비용이 늘고 응답 시간도 길어집니다.
실행 ①에서 출력한 응답 JSON 안의 `usage`가 이 두 숫자를 보여 주는 자리입니다.
사외(openai) 출력에서는 `prompt_tokens`가 입력 토큰 수, `completion_tokens`가 출력 토큰 수, `total_tokens`가 둘을 더한 수입니다.
사내 경로에서도 `usage`가 오는지는 아직 확인하지 않았습니다.

**되돌아보기: 실행 ①의 usage**

실행 ①의 결과로 돌아갑니다. API를 다시 호출하지 않습니다.
아래 셀은 실행 ①에서 받아 둔 응답(`recv_json`)에서 `usage` 부분만 다시 꺼내 보여 줍니다.

In [ ]:
from pprint import pprint

usage = ns01["recv_json"].get("usage")
if usage:
    pprint(usage)
else:
    print("이 환경에서는 usage가 제공되지 않습니다")

**볼 지점**
- 입력 토큰 수와 출력 토큰 수를 직접 읽어 적습니다 (p10).
- `prompt_tokens`가 입력 토큰, `completion_tokens`가 출력 토큰, `total_tokens`가 둘의 합입니다.
- 숫자는 실행할 때마다, 환경마다 다릅니다.
- "이 환경에서는 usage가 제공되지 않습니다"가 나오면 그대로 적습니다.

In [ ]:
show_slides("1-2", [11])

토큰 수를 줄이면 비용을 아낄 수 있습니다(p11).
필요한 정보만 골라 넣고, 반복되는 지시문과 대화 기록은 줄여 넣습니다.
검색한 문서도 전부 넣지 말고 관련 있는 일부만 넣습니다(p11).
답의 형식과 길이를 분명히 정하면 불필요한 출력이 줄어듭니다. 실습 질문의 "3문장으로"가 그 예입니다(p11).

In [ ]:
display_quiz("assets/quizzes/1-2/q22.json", max_width=800)

비용은 입력 토큰과 출력 토큰으로 정해지고, 입력에는 시스템 프롬프트와 이전 대화까지 들어갑니다(p10).
토큰을 아끼려면 필요한 것만 넣고, 답의 형식과 길이를 정합니다(p11).

## 3. Context Window
### 3-1 Context Window

**예측: 이전 대화를 다시 넣지 않으면?**

실행 ②에서는 먼저 "대한민국의 수도는 어디지?"라고 묻습니다. 다음 요청에서 "그곳의 인구는 어느정도야?"라고 묻습니다.
두 번째 요청에 이전 대화를 다시 넣지 **않으면**, 모델이 "그곳"을 서울로 알아들을까요? (p14)

1. 알아듣는다. 모델이 앞 대화를 기억한다.
2. 알아듣지 못한다. 이전 대화가 이번 요청에 없다.

아래 셀의 `None`을 1 또는 2로 바꾸고 실행합니다. 답은 실행 ② 단계 4 뒤에서 확인합니다.

In [ ]:
예측_3_1 = None  # 1 또는 2 로 바꾸세요

In [ ]:
show_slides("1-2", [12, 13, 14])

Context Window(컨텍스트 윈도우)는 모델이 한 번의 요청에서 읽고 고려할 수 있는 토큰 범위입니다(p13, p14).
그 안에는 시스템 메시지, 사용자 메시지, 이전 대화, 도구 실행 결과가 모두 들어갑니다(p14).
모델은 이번 요청에 들어 있지 않은 정보를 직접 참조하지 못합니다.
그래서 이전 대화를 기억하게 하려면 그 대화를 다음 요청에 다시 넣어야 합니다. 뒤의 실행 ②에서 02_openai.py 55~56줄의 `context +=`가 그 일을 합니다(p14).
긴 문서는 나눠서 필요한 부분만 찾아 넣고, 긴 대화는 요약하거나 중요한 상태만 따로 저장합니다(p14).

모델은 한 번의 요청에 들어 있는 토큰만 봅니다(p13, p14). 기억하게 하려면 이전 대화를 다시 넣어야 합니다(p14).

### ▶ 실행 ② `02_openai.py` — 한 번 묻기, 이어서 묻기, 조각으로 받기

이 파일은 OpenAI SDK(API를 쉽게 호출하도록 만든 파이썬 도구 모음)로 세 가지를 차례로 보여 줍니다. Single Turn(한 번 묻고 한 번 받기), Multi Turn(이어서 묻기), Streaming(조각으로 받기)입니다.
01과 달리 `client.responses.create(...)`를 쓰고, 보낼 내용을 `messages`가 아니라 `input`에 넣습니다.
사내 게이트웨이가 이 방식(Responses API)을 지원하는지는 아직 사내에서 확인하지 않았습니다.
사내에서 오류가 나면 이 실행은 건너뛰고, 코드 지도만 보고 넘어갑니다.

In [ ]:
show_code_map('../src/raw/Chapter01/01-llm-api/02_openai.py', [{'title': '준비와 환경 분기',
  'marker': 'import os',
  'note': '필요한 도구를 불러오고 .env를 읽습니다.\n'
          'LLM_PROVIDER 값에 따라 OpenAI 클라이언트(API에 요청을 보내는 도구)를 만듭니다.\n'
          '사내이면 사내 게이트웨이 주소와 헤더를 씁니다.'},
 {'title': 'Single Turn',
  'marker': '# ── Single Turn',
  'page': 7,
  'note': '질문 한 번, 답 한 번입니다.\ninput에 문자열 하나를 넣으면 사용자 메시지로 처리됩니다.\n답은 response.output_text로 꺼냅니다.'},
 {'title': 'Multi Turn 첫 질문',
  'marker': '# ── Multi Turn',
  'page': 18,
  'note': 'input에 역할이 붙은 메시지 목록(context)을 넣습니다.\n'
          'developer는 시스템 지시, user는 사용자 입력입니다.\n'
          '시스템 지시는 "단답형으로, 경어체로"입니다.'},
 {'title': '맥락 이어 붙이기',
  'marker': '# 이전 응답',
  'page': 14,
  'note': '첫 답(response_1.output)과 새 질문을 context 뒤에 붙입니다.\n'
          '늘어난 context 전체를 다시 보냅니다.\n'
          '그래서 "그곳"이 무엇인지 모델이 읽을 수 있습니다.'},
 {'title': 'Streaming',
  'marker': '# ── Streaming',
  'page': 6,
  'note': '답을 다 만든 뒤 한 번에 받지 않고, 만드는 대로 조각(이벤트)으로 받습니다.\n'
          '출력에는 response.output_text.delta라는 이름으로 나오는 이벤트에 텍스트 조각이 들어 있습니다.\n'
          '코드는 이벤트 이름에 output_text.delta가 들어 있는지로 골라냅니다(71줄).\n'
          '마지막에 이벤트 종류별 개수를 Counter로 셉니다.'},
 {'title': '참고 (실행 안 함)',
  'marker': '# ── Temperature',
  'page': 19,
  'note': 'temperature와 top_p 예시입니다.\n""" 안에 있어 실행되지 않습니다.'}], deck="1-2")

#### 실행 ② · 단계 1

01과 같은 방식으로 `LLM_PROVIDER`를 보고 연결할 곳을 고릅니다(02_openai.py의 10~28줄).
다만 주소와 헤더를 직접 조립하지 않고 `OpenAI(...)`로 `client`(요청을 대신 보내 주는 도구)를 만듭니다.
사내 경로에서 `api_key='samsung-gateway'`는 형식상 넣는 값이고, 실제 인증은 헤더에서 한다고 주석에 적혀 있습니다.

In [ ]:
ns02 = run_step('../src/raw/Chapter01/01-llm-api/02_openai.py', None, '# ── Single Turn')

**볼 지점**
- 이 단계는 출력이 없습니다. 오류 없이 끝나면 다음 단계로 갑니다.
- 오류 메시지가 나오면 실행 ① 단계 1과 같은 문구인지 확인합니다(15줄, 28줄).
- 이 파일은 Responses API(OpenAI의 새 요청 방식, 01의 `messages` 대신 `input`을 씀)를 사용합니다. 사내 게이트웨이가 이 방식을 지원하는지는 아직 확인하지 않았습니다. 오류가 나면 코드 지도만 보고 넘어갑니다.

#### 실행 ② · 단계 2

Single Turn은 질문 한 번, 답 한 번입니다(34~40줄).
`input`에 문자열 하나를 넣으면 사용자 메시지로 처리된다고 주석에 적혀 있습니다.
01에서 JSON 속 `choices[0].message.content`로 찾던 답을 여기서는 `response.output_text` 하나로 꺼냅니다.

In [ ]:
ns02 = run_step('../src/raw/Chapter01/01-llm-api/02_openai.py', '# ── Single Turn', '# ── Multi Turn')

**볼 지점**
- "LLM이 무엇인지 3문장으로 설명해줘."에 대한 답이 한 덩어리로 나오는지 봅니다(36~40줄).
- 실행 ①과 같은 질문입니다. 답 문장이 똑같은지 비교해 봅니다.

#### 실행 ② · 단계 3

이번에는 `input`에 역할이 붙은 메시지 목록 `context`를 넣습니다(47~51줄).
`developer` 메시지에는 "단답형으로, 경어체로" 답하라는 규칙이, `user` 메시지에는 "대한민국의 수도는 어디지?"가 들어 있습니다.
이 목록 `context`가 이번 요청의 컨텍스트입니다(p14).

In [ ]:
ns02 = run_step('../src/raw/Chapter01/01-llm-api/02_openai.py', '# ── Multi Turn', '# 이전 응답')

**볼 지점**
- "대한민국의 수도는 어디지?"에 대한 답이 짧은 존댓말로 나오는지 확인합니다(48~52줄).
- 짧은 존댓말로 답하라는 규칙은 `developer` 메시지(48줄)에 들어 있습니다. 코드 주석(46줄)은 이것을 "시스템 지시"라고 부릅니다.

#### 실행 ② · 단계 4

첫 답(`response_1.output`)과 새 질문 "그곳의 인구는 어느정도야?"를 `context` 뒤에 붙입니다(55~56줄).
그다음 늘어난 `context` 전체를 다시 보냅니다(57줄).
이전 대화를 다시 보내므로 두 번째 요청의 입력 토큰은 첫 요청보다 늘어납니다(p10).

In [ ]:
ns02 = run_step('../src/raw/Chapter01/01-llm-api/02_openai.py', '# 이전 응답', '# ── Streaming')

**볼 지점**
- "그곳의 인구는 어느정도야?"에 대한 답이 서울의 인구를 말하는지 살펴봅니다(56~58줄).
- 앞 질문과 답을 `context +=`로 다시 넣었기 때문에 "그곳"을 알아들을 수 있습니다(55~56줄) (p14).
- 붙인 첫 답(`response_1.output`)은 텍스트가 아니라 여러 정보가 든 묶음입니다. 그래서 코드 화면에 assistant 텍스트가 보이지 않습니다(55줄).
- 아래 셀에서 3-1의 예측과 비교합니다.

In [ ]:
print("두 번째 답:", ns02["response_2"].output_text)
print()
if 예측_3_1 is None:
    print("예측을 적지 않았습니다. 3-1의 셀에서 예측_3_1을 1 또는 2로 바꿔 실행한 뒤, 이 셀을 다시 실행하세요.")
elif 예측_3_1 == 2:
    print("예측 2번(알아듣지 못한다)은 p14의 설명과 같습니다.")
elif 예측_3_1 == 1:
    print("예측 1번(모델이 기억한다)은 p14의 설명과 다릅니다. 모델은 요청에 담긴 것만 읽습니다.")
else:
    print("예측_3_1에는 1 또는 2를 적으세요.")
print()
print("원본은 55~56줄에서 첫 답과 새 질문을 context에 붙여 다시 보냈습니다.")
print("위 답이 서울의 인구를 말한다면, 이전 대화가 요청에 들어 있었기 때문입니다 (p14).")
print("원본은 이전 대화를 넣지 않는 경우는 실행하지 않습니다.")

#### 실행 ② · 단계 5

`client.responses.stream(...)`은 답을 다 만든 뒤 한 번에 주지 않고, 만드는 대로 조각(이벤트)으로 보냅니다.
이벤트 이름에 `output_text.delta`가 들어 있으면(출력에서는 `response.output_text.delta`) 그 안의 텍스트 조각(`event.delta`)을 줄바꿈 없이 바로 출력합니다(71~72줄).
p6에서 본 "토큰을 하나씩 골라 붙인다"를 화면에서 보는 단계입니다(p6).
마지막에 `Counter`(개수를 세는 도구)로 어떤 종류의 이벤트가 몇 번 왔는지 셉니다.

In [ ]:
ns02 = run_step('../src/raw/Chapter01/01-llm-api/02_openai.py', '# ── Streaming', '# ── Temperature')

**볼 지점**
- 답변 텍스트가 한 번에 뜨지 않고 조각으로 이어서 나오는지 확인합니다(69~72줄) (p6).
- 맨 아래 `Counter(...)`에서 `response.output_text.delta` 옆의 수를 직접 읽습니다. 텍스트 조각 이벤트가 몇 번 왔는지입니다(74줄).
- 조각 수와 출력 토큰 수가 같다고 단정할 수는 없습니다. 조각이 차례로 쌓인다는 것만 봅니다.
- 수는 실행할 때마다 다릅니다.

Single Turn은 문자열 하나를, Multi Turn은 역할이 붙은 메시지 목록을 보냅니다.
이전 답을 `context`에 다시 넣어야 "그곳"을 이해하고(p14), Streaming은 답이 조각으로 쌓이는 모습을 보여 줍니다(p6).

### 3-2 입력 품질과 출력 품질

In [ ]:
show_slides("1-2", [15, 16])

LLM의 답 품질은 모델 성능뿐 아니라 컨텍스트에 어떤 정보가 들어갔는지에 크게 좌우됩니다(p15, p16).
목표가 분명하면 답의 방향이 안정되고, 배경 정보가 충분하면 추측과 환각(사실이 아닌 내용을 지어내는 것)이 줄어듭니다(p16).
반대로 불필요하거나 서로 부딪치는 정보가 많으면 판단이 흐려질 수 있습니다.
좋은 프롬프트는 답을 맞히라고 요구하지 않고, 답을 만들 수 있는 조건을 줍니다(p16).
실행 ②의 시스템 지시 "경어체, 단답형"은 출력 형식을 알려 준 예입니다.

In [ ]:
show_checklist(['역할과 목표를 분명히 준다', '필요한 배경 정보와 제약 조건을 함께 준다', '원하는 출력 형식과 평가 기준을 알려 준다', '모호한 지시나 서로 부딪히는 요구를 줄인다'], title='좋은 입력의 네 가지 (p16)')

답의 품질은 컨텍스트에 넣은 정보가 크게 좌우하고, 좋은 입력은 역할·목표, 배경·제약, 출력 형식, 모호함 줄이기를 갖춥니다(p16).
이 네 항목은 5절의 CodeMate 지시문 틀로 다시 씁니다.

## 4. LLM API의 대화 구조
### 4-1 System, User, Assistant 역할

In [ ]:
show_slides("1-2", [17, 18])

Chat 기반 LLM API는 대화를 역할이 붙은 메시지 목록으로 표현합니다(p18).
System 메시지는 모델의 역할, 규칙, 제약 조건을 정하고 User 메시지는 질문, 요청, 입력 데이터를 전합니다.
Assistant 메시지는 LLM이 만든 답이며, 다음 요청에 다시 넣어 맥락을 이어 줍니다(p18).
역할 구분은 모양만의 문제가 아니라, 모델이 지시의 우선순위와 대화 흐름을 읽는 기준입니다(p18).
실행 ②의 `context`가 바로 이 목록이고, 뒤의 실행 ③에서는 `SystemMessage`와 `HumanMessage`로 같은 구분을 합니다.

**되돌아보기: 실행 ② 단계 3의 코드 화면**

실행 ② 단계 3의 코드 화면으로 올라갑니다. 다시 실행하지 않습니다.
- `developer` 역할(48줄)이 p18의 System 자리입니다. 코드 주석(46줄)이 "developer: 시스템 지시"라고 적어 두었습니다 (p18).
- `user` 역할(49줄)이 User 메시지입니다 (p18).
- Assistant 메시지는 `response_1.output`을 통째로 붙인 것(55줄)이라 텍스트로는 보이지 않습니다 (p18).
- 실행 ①의 `01_requests.py`는 `user` 역할 하나만 씁니다(33줄).

In [ ]:
display_quiz("assets/quizzes/1-2/q41.json", max_width=800)

System은 규칙, User는 요청, Assistant는 모델의 답이며, 이 구분이 지시의 우선순위와 대화 흐름을 정합니다(p18).
코드에서는 System 자리에 `developer`(02)나 `SystemMessage`(03)를 씁니다.

### ▶ 실행 ③ `03_langchain.py` — LangChain과 `get_llm()`

이 파일은 같은 질문을 LangChain(LLM 앱을 만들 때 쓰는 파이썬 도구 모음)으로 보냅니다.
모델 준비는 `get_llm()` 한 줄이고, 어느 LLM으로 연결할지는 `src/config/llm.py`가 대신 고릅니다.
01·02는 이 분기를 파일마다 직접 썼지만 03은 이 함수 하나로 끝냅니다.
메시지는 p18의 System과 User를 `SystemMessage`와 `HumanMessage`로 나눠 적습니다(p18).
보내는 방법은 한 번에 받는 `invoke`와 조각으로 받는 `stream` 두 가지입니다.

In [ ]:
show_code_map('../src/config/llm.py', [{'title': '불러오기', 'marker': 'import os', 'note': '필요한 도구를 불러오고 .env를 읽습니다.'},
 {'title': '필요한 값 확인',
  'marker': 'def _required',
  'note': '환경변수 값을 읽습니다.\n비어 있으면 "환경변수가 필요합니다" 오류를 냅니다.'},
 {'title': 'get_llm() 시작',
  'marker': 'def get_llm',
  'note': 'LLM_PROVIDER 값을 읽습니다. 없으면 samsung입니다.\n모든 실습이 이 함수 하나로 모델을 준비합니다.'},
 {'title': '사외 OpenAI',
  'marker': 'if provider == "openai":',
  'note': 'OPENAI_MODEL과 OPENAI_API_KEY로 ChatOpenAI를 만듭니다.\n모델이 gpt-6-luna이면 추론 설정을 추가합니다.'},
 {'title': '사내 게이트웨이',
  'marker': 'elif provider == "samsung":',
  'note': 'config/samsung.py에서 사내 주소·모델·인증 헤더를 받아 ChatOpenAI를 만듭니다.\n'
          '추론 기능(enable_thinking)을 끄는 값을 함께 보냅니다.'},
 {'title': '그 밖의 값', 'marker': 'raise ValueError', 'note': 'openai나 samsung이 아니면 오류를 냅니다.'}])

In [ ]:
show_code_map('../src/raw/Chapter01/01-llm-api/03_langchain.py', [{'title': '불러오기',
  'marker': 'from dotenv import load_dotenv',
  'note': '필요한 도구를 불러오고 .env를 읽습니다.\n모델 준비에 쓸 get_llm()도 불러옵니다.'},
 {'title': '모델 준비',
  'marker': '# 핵심은',
  'page': 18,
  'note': '원본에 남은 메모(8~11줄)는 사용자가 묻고 싶은 것과 시스템으로 지시하고 싶은 것을 나눠야 한다고 말합니다. p18의 User·System 구분과 같은 '
          '말입니다.\n'
          'llm = get_llm() 한 줄로 모델을 준비합니다.'},
 {'title': '기본 invoke',
  'marker': '# ── 기본 invoke',
  'page': 18,
  'note': 'SystemMessage에 역할("초보자를 위한 AI 튜터")을, HumanMessage에 질문을 넣습니다.\n'
          'invoke()는 답을 다 만든 뒤 한 번에 돌려줍니다.\n'
          '답변 텍스트는 response.content로 꺼냅니다.'},
 {'title': 'Streaming',
  'marker': '# ── Streaming',
  'page': 6,
  'note': 'stream()은 답을 조각으로 하나씩 돌려줍니다.\n'
          "주석은 end=''로 줄바꿈 없이 출력한다고 적었지만 코드는 print(chunk)로 조각을 통째로 한 줄씩 출력합니다."}], deck="1-2")

#### 실행 ③ · 단계 1

`SystemMessage`, `HumanMessage`, `get_llm`을 불러오고, 12줄의 `get_llm()`으로 `llm`을 만듭니다.
그래서 이 파일에는 주소나 키를 고르는 줄이 없습니다.
8~11줄은 코드가 아니라 원본에 남아 있는 메모(주석, `#`로 시작해 실행되지 않는 줄)입니다.

In [ ]:
ns03 = run_step('../src/raw/Chapter01/01-llm-api/03_langchain.py', None, '# ── 기본 invoke')

**볼 지점**
- 이 단계는 출력이 없습니다. 오류 없이 끝나면 모델 준비가 된 것입니다.
- 코드 화면의 8~11줄 메모를 읽습니다. "사용자가 질문하고 싶은 것, 시스템적으로 지시하고 싶은 것"을 나눠야 한다고 적혀 있습니다.
- 이것은 p18의 User 메시지와 System 메시지 구분과 같은 말입니다 (p18).

#### 실행 ③ · 단계 2

`messages`에 `SystemMessage`와 `HumanMessage`를 차례로 넣습니다(17~21줄). 02에서 `developer`로 쓰던 자리를 여기서는 `SystemMessage`로 씁니다(p18).
`llm.invoke(messages)`는 답을 다 만든 뒤 한 번에 돌려줍니다.
24줄 메모에는 돌아온 것이 `AIMessage`(AI 응답 메시지)라고 적혀 있습니다. 답변 텍스트는 25줄에서 `.content`로 꺼냅니다.

In [ ]:
ns03 = run_step('../src/raw/Chapter01/01-llm-api/03_langchain.py', '# ── 기본 invoke', '# ── Streaming')

**볼 지점**
- `SystemMessage`(19줄)에는 "너는 초보자를 위한 AI 튜터야"가, `HumanMessage`(20줄)에는 질문이 들어 있습니다.
- `SystemMessage`가 p18의 System, `HumanMessage`가 User 메시지입니다 (p18).
- 출력된 답이 초보자에게 설명하듯 쓰였는지 확인합니다. 실행 ①·②의 답과 말투를 비교해 봅니다.

#### 실행 ③ · 단계 3

`llm.stream(messages)`는 답을 조각으로 하나씩 돌려줍니다(31줄).
29줄 주석은 `end=''`로 줄바꿈 없이 출력한다고 하지만, 32줄 코드는 `print(chunk)`로 조각을 통째로 출력합니다. 이 차이는 5절 실습 B에서 다룹니다.
조각 안의 `usage_metadata`는 실행 ①의 `usage`와 같은 토큰 수 정보이고, 이름만 `input_tokens`(입력), `output_tokens`(출력)로 다릅니다(p10).

**실행 전에 읽습니다.** 다음 셀은 출력이 깁니다. 각 줄의 `content=` 뒤 텍스트만 봅니다.

In [ ]:
ns03 = run_step('../src/raw/Chapter01/01-llm-api/03_langchain.py', '# ── Streaming', None)

**볼 지점**
- 마지막 부분 줄에 `usage_metadata`가 있으면 그 안의 `input_tokens`(입력 토큰)와 `output_tokens`(출력 토큰)를 직접 읽습니다. 실행 ①의 `prompt_tokens`·`completion_tokens`와 같은 뜻입니다 (p10).
- 사내 환경에서 `usage_metadata`가 나오는지는 아직 확인하지 않았습니다. 없으면 "이 환경에서는 제공되지 않음"으로 적습니다.
- 조각이 한 줄에 하나씩 출력됩니다(31~32줄).
- 각 줄의 `content=` 뒤 텍스트를 위에서 아래로 이어 읽으면 답 문장이 됩니다 (p6).
- 조각 줄 수와 출력 토큰 수가 같다고 단정할 수는 없습니다.

03은 `get_llm()` 한 줄로 모델을 준비하고, `SystemMessage`·`HumanMessage`로 역할을 나눠 `invoke`와 `stream`으로 보냅니다(p18).
같은 질문을 01은 요청을 직접 조립해, 02는 OpenAI SDK로, 03은 LangChain으로 보냈습니다.

### 4-2 모델 파라미터의 영향

In [ ]:
show_slides("1-2", [19])

모델 파라미터(모델에 함께 보내는 설정값)는 같은 입력을 넣었을 때 답이 얼마나 다양하고 예측 가능할지를 조정합니다(p19).
Temperature는 다음 토큰의 확률을 조정하고 Top-p는 후보 토큰의 누적 확률 한도를 조정합니다(p19).
둘 다 낮으면 일관되고 보수적인 답에, 높으면 다양한 표현과 아이디어에 유리합니다.
너무 높으면 가능성이 거의 없는 토큰이 나올 수 있어 주의합니다(p19).
p6에서 "후보 중 하나를 고른다"고 했는데, 이 고르는 방식을 바꾸는 값이 이 두 설정입니다(p6, p19).

**코드에서 찾아보기** (실행하지 않습니다)
- `01_requests.py`의 요청 본문에 `"temperature": 0.5`가 들어 있습니다(35줄).
- 사외 실행에서 모델이 `gpt-6-luna`이면 42줄에서 temperature가 빠진 채 전송되고, 사내 경로에서는 0.5가 그대로 전송됩니다(`01_requests.py` 38~43줄).
- `02_openai.py` 끝의 temperature·top_p 예시(81~89줄)는 `"""` 안에 있어 실행되지 않습니다.

In [ ]:
display_quiz("assets/quizzes/1-2/q42.json", max_width=800)

Temperature와 Top-p는 다음 토큰을 고르는 방식을 바꾸며, 낮으면 일관되고 높으면 다양합니다(p19).
실습 코드에서 실제로 전송되는 값은 01의 `temperature: 0.5`뿐이고(모델이 `gpt-6-luna`이면 빠짐), 02 끝의 예시는 실행되지 않는 참고용입니다.

## 5. CodeMate 실습

이 절에서는 CodeMate(사내 Coding AI)에게 지시문을 주고, 돌아온 답을 원본 코드와 맞춰 보며 검토합니다.
지시문은 3-2에서 본 p16의 네 항목을 틀로 씁니다. 역할·목표, 배경·제약, 출력 형식, 모호함 줄이기입니다(p16).
여기에 **하지 말 것**과 **왜 이렇게 했는지 3줄로 설명**을 추가합니다.
세 실습 모두 목표 코드는 원본입니다.
A와 B는 필수이고, C는 먼저 끝난 사람이 하는 선택 실습입니다.

**실습 A (필수): 설명 받기, 코드 생성 금지**

CodeMate(사내 Coding AI)에게 `01_requests.py`의 30~54줄을 설명받습니다. 목표 코드는 원본입니다.
지시문은 3-2에서 본 p16의 네 항목(역할·목표, 배경·제약, 출력 형식, 모호함 줄이기)에 "하지 말 것"과 "왜 이렇게 했는지 3줄"을 더한 틀입니다 (p16).
아래 지시문을 복사해 CodeMate에 붙여 넣습니다.

```text
[역할·목표]
당신은 파이썬을 처음 배우는 사람을 돕는 코드 해설자입니다.
목표는 제가 src/raw/Chapter01/01-llm-api/01_requests.py 의 30~54줄을 이해하는 것입니다.

[배경·제약]
- 이 파일은 LLM API에 요청을 보내고 응답 JSON을 출력합니다.
- 저는 이 파일을 이미 한 번 실행했고, 출력은 따로 가지고 있습니다.
- 30~54줄만 다룹니다. 1~29줄과 57줄 아래의 """ 안 코드는 설명하지 않습니다.

[출력 형식]
1. 30~54줄을 한 줄씩(빈 줄 제외) "줄 번호 | 코드 | 쉬운 한국어 설명 1문장" 표로 써 주세요.
2. 52줄 pprint 출력의 usage 부분에 어떤 이름의 값들이 나올지 예측해 목록으로 써 주세요.
   숫자는 맞히지 말고, 각 값이 무엇을 세는지 한 문장씩 붙여 주세요.

[모호함 줄이기]
- 전문 용어는 처음 나올 때 괄호 안에 쉬운 말로 풀어 주세요.
- 확실하지 않은 것은 "확실하지 않음"이라고 써 주세요. 추측을 사실처럼 쓰지 마세요.

[하지 말 것]
- 코드를 새로 쓰거나 고치지 마세요. src/ 아래 파일을 고치지 마세요. 파일을 만들거나 저장하지 마세요.
- 더 좋은 코드나 리팩터링을 제안하지 마세요.

마지막에 왜 이렇게 설명했는지 3줄로 써 주세요.
```

CodeMate의 usage 예측을 실행 ① 단계 3의 출력과 대조합니다. 아래 체크리스트로 답을 검토합니다.

In [ ]:
show_checklist(['30~54줄만 설명했다. 다른 줄이나 새 코드가 없다',
 '표의 줄 번호와 코드가 원본 01_requests.py와 맞다',
 "usage 예측을 실행 ① 단계 3의 출력과 대조했다 (없으면 '이 환경에서는 제공되지 않음')",
 '확실하지 않은 내용을 사실처럼 쓴 곳이 없다',
 'src/ 아래 파일이 바뀌지 않았다',
 "마지막에 '왜 이렇게 설명했는지' 3줄이 있다"], title='실습 A 리뷰')

**실습 B (필수): 원본끼리 비교**

raw 실습본과 정답본의 `03_langchain.py`는 스트리밍 결과를 출력하는 방식이 다릅니다. raw는 `print(chunk)`, 정답본은 `print(chunk.content, end='', flush=True)`입니다.
CodeMate에게 두 방식을 비교하게 하고, 그 답을 실행 ③ 단계 3의 출력과 맞춰 봅니다.

```text
[역할·목표]
당신은 강의 화면을 검토하는 코드 리뷰어입니다.
목표는 두 원본 파일의 Streaming 부분이 화면에 무엇을 출력하는지, 그 차이를 제가 이해하는 것입니다.

[배경·제약]
- 파일 1: src/raw/Chapter01/01-llm-api/03_langchain.py 의 27~32줄
- 파일 2: src/solution/Chapter01/01-llm-api/03_langchain.py 의 23~28줄
- 두 파일 모두 SystemMessage와 HumanMessage 목록으로 llm.stream()을 호출합니다. 메시지 문구는 조금 다릅니다.
- 이 두 구간만 비교합니다. 나머지 줄은 다루지 않습니다.
- 저는 코딩이 익숙하지 않습니다. 쉬운 말로 써 주세요.

[출력 형식]
1. 두 방식이 화면에 무엇을 출력하는지 각각 2문장으로 써 주세요.
2. "파일 1에서만 보이는 정보"와 "파일 2에서만 보이는 정보"를 각각 목록으로 써 주세요.
3. 강의 화면(뒷자리에서도 읽어야 함)에는 어느 쪽이 나은지 고르고, 이유를 2문장으로 써 주세요.

[모호함 줄이기]
- "조각(chunk)"은 스트리밍으로 한 번에 받는 응답 일부를 뜻합니다. 이 뜻으로만 써 주세요.
- 주석이 아니라 실제 코드 줄을 기준으로 비교해 주세요. 주석과 코드가 다르면 그 점도 적어 주세요.
- 실행하지 않고는 알 수 없는 것은 "실행해서 확인 필요"라고 써 주세요.

[하지 말 것]
- src/ 아래 파일(두 파일 포함)을 고치거나 새 파일을 만들지 마세요.
- 두 방식을 섞은 새 코드를 제안하지 마세요.

마지막에 왜 이렇게 판단했는지 3줄로 써 주세요.
```

raw 쪽 "파일 1에서만 보이는 정보"에 `usage_metadata` 같은 조각 속 정보가 나오면, 실행 ③ 단계 3의 출력에 정말 있는지 확인합니다.

In [ ]:
show_checklist(['두 파일의 줄 범위를 맞게 읽었다 (raw 27~32줄, 정답본 23~28줄)',
 "raw의 주석(end='')과 실제 코드(print(chunk))가 다르다는 점을 짚었다",
 "'파일 1에서만 보이는 정보'를 실행 ③ 단계 3의 출력과 대조했다",
 '실행해 보지 않고 단정한 곳이 없다',
 'src/ 아래 파일이 바뀌지 않았다',
 "마지막에 '왜 이렇게 판단했는지' 3줄이 있다"], title='실습 B 리뷰')

**실습 C (선택, 먼저 끝난 사람): 지시로 다시 만들고 원본과 비교**

원본 `01_requests.py`의 동작을 말로 적은 지시문입니다. CodeMate가 이 지시만 보고 같은 파일을 만들게 합니다. 목표 코드는 원본입니다.
CodeMate가 `lessons/work/`에 파일을 만들 수 있는지는 사내 환경에서는 아직 확인하지 않았습니다.

```text
[역할·목표]
당신은 제 지시대로 파이썬 파일을 만드는 페어 프로그래머입니다.
목표는 아래 동작을 그대로 하는 파일 lessons/work/01_requests_mine.py 를 만드는 것입니다.

[배경·제약]
- 라이브러리는 os, pprint, requests, python-dotenv만 씁니다.
- python-dotenv의 load_dotenv()로 .env를 읽은 뒤 LLM_PROVIDER 값을 읽습니다. 없으면 'samsung'입니다. 앞뒤 공백을 지우고 소문자로 바꿉니다.
- 'openai'이면:
  - 주소는 https://api.openai.com/v1/chat/completions 입니다.
  - OPENAI_MODEL, OPENAI_API_KEY를 읽습니다. 둘 중 하나라도 없으면 RuntimeError를 냅니다.
  - 헤더는 Authorization(Bearer 키)과 Content-Type(application/json)입니다.
- 'samsung'이면:
  - config.samsung 의 llm_config()를 호출해 base_url, model, headers를 받습니다.
  - 주소는 base_url 끝의 / 를 지우고 /chat/completions 를 붙입니다.
  - 헤더는 받은 headers에 Content-Type(application/json)을 더합니다.
- 그 밖의 값이면 ValueError를 냅니다.
- 요청 본문: model, messages(user 역할 1개, 내용 "LLM이 무엇인지 3문장으로 설명해줘."), temperature 0.5, stream False.
- samsung이면 본문에 include_reasoning False를 추가합니다.
  아니고 모델이 gpt-6-luna이면 temperature를 빼고 reasoning_effort 'none'을 추가합니다.
- POST로 보냅니다. timeout은 30초입니다. 서버가 오류로 답하면 예외를 냅니다.

[출력 형식]
1. 파일 하나만 만듭니다. 위 순서대로 씁니다.
2. 실행하면 "### 응답 Json" 다음 줄에 응답 JSON 전체를 pprint로, "### 결과 메시지" 다음 줄에 choices[0].message.content 를 출력합니다.
3. 파일을 만든 뒤, 각 제약이 몇 번째 줄에 있는지 표로 써 주세요.

[모호함 줄이기]
- 위에 없는 동작은 넣지 마세요. 헷갈리면 만들기 전에 저에게 물어 주세요.

[하지 말 것]
- src/ 아래 파일을 열어 베끼거나 고치지 마세요.
- 키 값을 코드에 쓰거나 화면에 출력하지 마세요.
- 재시도, 로그, 함수 나누기 같은 기능을 더하지 마세요.

마지막에 왜 이렇게 만들었는지 3줄로 써 주세요.
```

**검토 방법**
1. VS Code 탐색기에서 원본 `src/raw/Chapter01/01-llm-api/01_requests.py`와 만든 파일을 함께 선택하고 "Compare Selected(선택 항목 비교)"로 나란히 봅니다.
2. 다른 줄을 적습니다.
3. 빠진 조건이 있으면 지시문에 보강해 다시 시킵니다.

## 6. 정리 퀴즈

In [ ]:
display_quiz("assets/quizzes/1-2/final.json", max_width=800)

**이 노트북에서 본 것**
- 사전 학습만 한 모델은 그럴듯하게 이어 쓰고 API로 쓰는 모델은 대체로 지시 수행을 강화한 모델입니다 (p4, p5).
- 모델은 토큰을 하나씩 예측해 붙이고, API는 요청 → 후보 계산 → 누적 → 반환 순서로 답을 돌려줍니다 (p6, p7).
- 비용은 입력 토큰과 출력 토큰으로 정해지고, 입력에는 시스템 프롬프트와 이전 대화까지 들어갑니다 (p10).
- 모델은 요청에 담긴 컨텍스트만 읽습니다. 이전 대화도 요청에 다시 넣어야 합니다 (p14).
- 좋은 입력은 역할·목표, 배경·제약, 출력 형식을 주고 모호함을 줄입니다 (p16).
- 역할 구분(System, User, Assistant)은 모델이 지시의 우선순위와 대화 흐름을 해석하는 기준입니다 (p18).
- Temperature와 Top-p가 낮으면 일관된 답, 높으면 다양한 답에 유리합니다 (p19).